In [ ]:
!pip install -q chromadb sentence-transformers scikit-learn matplotlib

In [ ]:
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

In [ ]:
embed = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print('Embedding dim:', embed.get_sentence_embedding_dimension())

In [ ]:
paragraphs = [
    "Operating Systems: Process management and scheduling algorithms.",
    "Memory management: paging, segmentation, and virtual memory.",
    "File systems: directory structures and file allocation methods.",
    "Synchronization: semaphores, monitors, and critical sections.",
    "Deadlocks: prevention, avoidance, detection, and recovery.",
    "Database Management Systems: ER model and normalization.",
    "SQL: joins, subqueries, views, and triggers.",
    "Transactions: ACID properties and concurrency control.",
    "Indexing: B+ trees and hashing techniques.",
    "Computer Networks: routing algorithms and network layers."
]

print(f'Loaded {len(paragraphs)} paragraphs')

for i, p in enumerate(paragraphs):
    print(f'[{i+1}] {p}')

In [ ]:
client = PersistentClient(path='./chroma_db')

col = client.get_or_create_collection('hello_syllabus')

# Convert paragraphs into embeddings
vectors = embed.encode(paragraphs).tolist()

# Store inside ChromaDB
col.add(
    documents=paragraphs,
    embeddings=vectors,
    ids=[f'p{i}' for i in range(len(paragraphs))]
)

print(f'Indexed {col.count()} documents')

In [ ]:
queries = [
    'what is dynamic programming?',
    'machine learning topics',
    'operating system processes',
]

for q in queries:

    print(f'\nQuery: {q}')

    # Convert query into embedding
    qv = embed.encode([q]).tolist()

    # Search nearest matches
    results = col.query(
        query_embeddings=qv,
        n_results=3
    )

    docs = results['documents'][0]
    distances = results['distances'][0]

    for j, (d, dist) in enumerate(zip(docs, distances)):

        print(f'[{j+1}] (dist={dist:.3f}) {d}')

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Convert vectors list into numpy array
vectors_arr = np.array(vectors)

# Reduce dimensions: 384 → 2
pca = PCA(n_components=2)

xy = pca.fit_transform(vectors_arr)

# Create plot
plt.figure(figsize=(10, 8))

plt.scatter(xy[:, 0], xy[:, 1], s=100, alpha=0.6)

# Add labels
for i, p in enumerate(paragraphs):

    label = p[:30] + '...'

    plt.annotate(label, (xy[i, 0], xy[i, 1]), fontsize=8)

plt.title('Syllabus Paragraph Embeddings (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
# Add unrelated paragraph
outlier = "Today's special at the cafeteria is butter chicken with rice and naan."

# Store it in ChromaDB
col.add(
    documents=[outlier],
    embeddings=embed.encode([outlier]).tolist(),
    ids=['outlier_food']
)

# Reload all documents + embeddings
all_docs = col.get(include=['embeddings', 'documents'])

all_vecs = np.array(all_docs['embeddings'])

labels = all_docs['ids']

# PCA again
pca = PCA(n_components=2)

xy = pca.fit_transform(all_vecs)

# Plot
plt.figure(figsize=(10, 8))

colors = ['red' if 'outlier' in l else 'blue' for l in labels]

plt.scatter(xy[:, 0], xy[:, 1], c=colors, s=100, alpha=0.6)

# Labels
for i, l in enumerate(labels):

    short = labels[i] if 'outlier' in labels[i] else all_docs['documents'][i][:30] + '...'

    plt.annotate(short, (xy[i, 0], xy[i, 1]), fontsize=8)

plt.title('With Outlier (Red)')

plt.show()